# Weekly Project 5 - 3D point cloud processing 1
## Implementation of global registration 
### Task 1
Today, your task is to implement a global registration algorithm.

It should be able to roughly align two point clouds.
Implement the global registration, and then try the following:

1. Can you fit `r1.pcd` and `r2.pcd`?
2. Can you fit `car1.ply` and `car2.ply`?
The corresponding files are in the `global_registration` folder.


### Task 2 (Challange)
Challanges attempt either or both:
- Implement local registration.

- Attempt to reconstruct the car from the images in `car_challange` folder.

You can use the notebooks from Monday as a starting point.

## Weekly Project 05 - Start Here

In [1]:
import open3d as o3d
import numpy as np
import copy

In [2]:
# helper function for drawing (from monday's exercise)
def draw_registrations(source, target, transformation = None, recolor = False):
    source_temp = copy.deepcopy(source)
    target_temp = copy.deepcopy(target)
    if(recolor):
        source_temp.paint_uniform_color([1, 0.706, 0])
        target_temp.paint_uniform_color([0, 0.651, 0.929])
    if(transformation is not None):
        source_temp.transform(transformation)
    o3d.visualization.draw_geometries([source_temp, target_temp])

### Task 1

In [7]:
source = o3d.io.read_point_cloud("global_registration/r1.pcd")
target = o3d.io.read_point_cloud("global_registration/r2.pcd")

voxel_size = 0.05

#downsampling the point clouds
source_down = source.voxel_down_sample(voxel_size)
target_down = target.voxel_down_sample(voxel_size)

# FPFH feature extraction
radius_normal = voxel_size * 2
radius_feature = voxel_size * 5

source_down.estimate_normals(
    o3d.geometry.KDTreeSearchParamHybrid(radius=radius_normal, max_nn=30))
target_down.estimate_normals(
    o3d.geometry.KDTreeSearchParamHybrid(radius=radius_normal, max_nn=30))

source_fpfh = o3d.pipelines.registration.compute_fpfh_feature(
    source_down,
    o3d.geometry.KDTreeSearchParamHybrid(radius=radius_feature, max_nn=100))
target_fpfh = o3d.pipelines.registration.compute_fpfh_feature(
    target_down,
    o3d.geometry.KDTreeSearchParamHybrid(radius=radius_feature, max_nn=100))

# RANSAC based global registration
distance_threshold = voxel_size * 1.5
point_to_point = o3d.pipelines.registration.TransformationEstimationPointToPoint(False)
point_to_plane = o3d.pipelines.registration.TransformationEstimationPointToPlane()

ransac_result = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
    source_down, target_down,
    source_fpfh, target_fpfh,
    True,
    distance_threshold,
    point_to_point)

print(ransac_result)
print("Transformation is:")
print(ransac_result.transformation)

draw_registrations(source, target, ransac_result.transformation, recolor = True)

RegistrationResult with fitness=6.724790e-01, inlier_rmse=2.632278e-02, and correspondence_set size of 3201
Access transformation to get result.
Transformation is:
[[-0.52601114  0.84982965  0.033194    0.55361701]
 [-0.20986063 -0.16752055  0.96327326  0.86431178]
 [ 0.82417886  0.49972635  0.26646346 -1.44508552]
 [ 0.          0.          0.          1.        ]]


In [8]:
source = o3d.io.read_point_cloud("global_registration/car1.ply")
target = o3d.io.read_point_cloud("global_registration/car2.ply")

voxel_size = 0.05

#downsampling the point clouds
source_down = source.voxel_down_sample(voxel_size)
target_down = target.voxel_down_sample(voxel_size)

# FPFH feature extraction
radius_normal = voxel_size * 2
radius_feature = voxel_size * 5

source_down.estimate_normals(
    o3d.geometry.KDTreeSearchParamHybrid(radius=radius_normal, max_nn=30))
target_down.estimate_normals(
    o3d.geometry.KDTreeSearchParamHybrid(radius=radius_normal, max_nn=30))

source_fpfh = o3d.pipelines.registration.compute_fpfh_feature(
    source_down,
    o3d.geometry.KDTreeSearchParamHybrid(radius=radius_feature, max_nn=100))
target_fpfh = o3d.pipelines.registration.compute_fpfh_feature(
    target_down,
    o3d.geometry.KDTreeSearchParamHybrid(radius=radius_feature, max_nn=100))

# RANSAC based global registration
distance_threshold = voxel_size * 1.5
point_to_point = o3d.pipelines.registration.TransformationEstimationPointToPoint(False)
point_to_plane = o3d.pipelines.registration.TransformationEstimationPointToPlane()

ransac_result = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
    source_down, target_down,
    source_fpfh, target_fpfh,
    True,
    distance_threshold,
    point_to_point)

print(ransac_result)
print("Transformation is:")
print(ransac_result.transformation)

draw_registrations(source, target, ransac_result.transformation, recolor = True)

RegistrationResult with fitness=9.922636e-01, inlier_rmse=3.070664e-02, and correspondence_set size of 3463
Access transformation to get result.
Transformation is:
[[-0.01321647  0.05889735  0.99817655  2.02484365]
 [ 0.99988841  0.00773033  0.01278301  2.01189735]
 [-0.00696335  0.99823411 -0.05899294 -0.1087983 ]
 [ 0.          0.          0.          1.        ]]


### Task 2 (Challange)